# DH-FDM-PINN plate bending under a SYMMETRIC TRIANGULAR load, (FFFF).

In [1]:
"""
DH-FDM-PINN (Deterministic-Harmonic + FDM-PINN backbone) for Kirchhoff-Love
plate bending under a SYMMETRIC TRIANGULAR (tent) load, ALL FOUR EDGES
CLAMPED / FIXED (FFFF).

    q(x) = q0 * (2x/a)         for 0 <= x <= a/2
    q(x) = q0 * (2(a-x)/a)     for a/2 <= x <= a

Direct extension of dh_fdm_pinn_triangular_hardBC.py (the SS-SS-SS-SS
version). ONLY the boundary condition changes. The load, the domain, the
non-dimensionalization, the harmonic embedding, the FDM residual stencil,
the network backbone, the optimizer/schedule, gradient clipping,
diagnostics, and the output/plotting structure are all UNCHANGED -- see
that script's docstring for the load derivation, the harmonic-embedding
rationale, and the FD-stencil derivation/validation. This docstring only
covers what's different for FFFF.

-----------------------------------------------------------------------------
WHAT "FFFF" MEANS AND WHY THE BENCHMARK CHANGES
-----------------------------------------------------------------------------
FFFF = all four edges clamped (built-in / fixed): on every edge, BOTH the
deflection AND the slope normal to the edge vanish --

    w = 0  and  dw/dn = 0   on x=0, x=a, y=0, y=b

(SS-SS-SS-SS only required w=0 and the moment Mn=0; FFFF is stiffer -- it
also kills the rotation at the edge.)

THIS BREAKS THE NAVIER SERIES. Navier's double sine series
w = Sum W_mn sin(m*pi*x/a) sin(n*pi*y/b) is only usable because
sin(m*pi*x/a) simultaneously (a) diagonalizes the biharmonic operator
d^4/dx^4 term-by-term and (b) satisfies the plate's boundary conditions
(w=0 and Mx=0, i.e. w=0 and w_xx=0, at x=0,a). For a clamped edge the
second condition is w=0 AND w_x=0 -- no single sine (or cosine) mode
satisfies both simultaneously, so no finite or infinite sine/cosine series
in the Navier style can represent a clamped-plate solution exactly. This
is a standard, well-known fact in plate theory (Timoshenko & Woinowsky-
Krieger, "Theory of Plates and Shells" -- clamped rectangular plates have
no closed-form Navier solution; classical treatments resort to Levy-type
superposition, tabulated series-of-series coefficients, or Ritz/Galerkin
methods). There is consequently no simple closed-form "ground truth" to
drop in here the way analytical_w_nd() worked for SS-SS-SS-SS.

WHAT'S USED INSTEAD: RAYLEIGH-RITZ WITH CLAMPED-CLAMPED BEAM FUNCTIONS.
This is the classical remedy (see e.g. Young, 1950; Leissa's plate/shell
vibration monographs; Timoshenko & Woinowsky-Krieger Sec. 39) and is what
"deriving the analytical solution" for FFFF actually means in practice:
build a basis that satisfies the clamped BC exactly, then solve a (small,
dense) linear system for the coefficients by Galerkin-projecting the plate
PDE onto that basis. It converges to the true solution as the number of
modes grows, exactly analogous to how the SS truncated series converges as
M_TERMS_TRUNC, N_TERMS_TRUNC grow -- the difference is the SS system is
diagonal (closed-form coefficients) while the FFFF system is a dense
K^2 x K^2 linear system (solved once, numerically, at import time).

Step by step:

1. CLAMPED-CLAMPED BEAM EIGENFUNCTIONS. On xi in [0,1], solve
   phi'''' = beta^4 * phi  with  phi(0)=phi'(0)=phi(1)=phi'(1)=0.
   The general solution satisfying phi(0)=phi'(0)=0 is
       phi(xi) = cosh(b xi) - cos(b xi) - sigma*(sinh(b xi) - sin(b xi))
   and phi(1)=0 forces sigma = (cosh(b)-cos(b))/(sinh(b)-sin(b)); phi'(1)=0
   then reduces (after eliminating sigma, see chat for the algebra) to the
   classical clamped-clamped characteristic equation
       cos(b) * cosh(b) = 1
   whose positive roots b_1 < b_2 < ... are solved here by Newton's method
   seeded at (2n+1)*pi/2 (an excellent initial guess for every n, including
   n=1 -- verified against the standard tabulated roots b_1=4.7300,
   b_2=7.8532, b_3=10.9956, b_4=14.1372, ... to solver tolerance).

2. ORTHOGONALITY (used to keep the system as sparse as it can be). Because
   d^4/dxi^4 with these BCs is self-adjoint and phi_i''''=b_i^4 phi_i, the
   phi_i are orthogonal in the plain L2 sense: int phi_i phi_j dxi =
   N_i delta_ij, and (via one integration by parts each way, boundary terms
   vanishing because phi(0)=phi(1)=0) int phi_i'' phi_j'' dxi = b_i^4 N_i
   delta_ij as well -- i.e. same-derivative-order overlaps ARE diagonal.
   What is NOT diagonal is the mixed overlap B_ik = int phi_i'' phi_k dxi
   (= int phi_i phi_k'' dxi, symmetric, by the same integration-by-parts
   argument) -- this is exactly the term the 2D plate's cross-derivative
   w_xxyy introduces, and it's what makes the clamped problem harder than
   the simply-supported one. N_i and B_ik have no simple closed form here
   and are computed by fine-grid numerical quadrature (trapezoidal rule,
   8001 points on [0,1] -- smooth analytic integrands, converged far beyond
   what's needed).

3. RITZ EXPANSION AND GALERKIN SYSTEM. Since a_nd=b_nd=1 (unit square in
   non-dimensional coordinates), write
       w(x,y) = Sum_{k,l=1..K} C_kl * phi_k(x) * phi_l(y)
   which satisfies w=0 and dw/dn=0 on all 4 edges term-by-term, exactly,
   for ANY choice of C_kl. Substituting into D*(w_xxxx+2 w_xxyy+w_yyyy)=q
   and Galerkin-projecting onto phi_k(x)phi_l(y) gives, using the
   orthogonality above,
       C_kl * N_k*N_l*(b_k^4+b_l^4)  +  2 * Sum_ij B_ik*B_jl*C_ij  =  Q_kl
   where Q_kl = int int q(x,y) phi_k(x) phi_l(y) dx dy is the load's modal
   projection (computed in closed form as a product Q_kl = q0 * I_k * J_l
   since the tent load separates: I_k against the tent shape in x, J_l =
   int phi_l dy against the uniform-in-y shape). This is a linear system of
   K^2 equations in the K^2 unknowns C_kl (dense, because of the B_ik B_jl
   coupling term -- unlike the SS case it does NOT collapse to a per-mode
   formula); it is assembled as a (K^2, K^2) matrix via
   `2*np.einsum('ik,jl->klij', B, B).reshape(K*K,K*K) + diag(D.flatten())`
   and solved once with `np.linalg.solve`.

4. VALIDATION (see chat). Before trusting this for the tent load, the same
   machinery was run for a UNIFORM load on a clamped square plate, whose
   center-deflection coefficient is a well-known tabulated benchmark
   (Timoshenko & Woinowsky-Krieger: w_center = 0.00126 * q*a^4/D). The Ritz
   solve above reproduces 0.0012651 at K_MODES=10 (0.0012670 at K=6,
   0.0012652 at K=12, 0.0012652 at K=14 -- converged to 5-6 significant
   figures by K=10), matching the textbook value to its stated precision.
   K_MODES=10 is used below: convergence is essentially complete by then,
   and it stays comfortably inside float64 precision (beta_10 ~= 33, so
   cosh/sinh terms reach ~1e14 -- pushing K much higher starts trading
   away precision to catastrophic cancellation in phi() near xi=1 for
   high-order modes, since phi(1)=0 exactly but is computed there as a
   difference of two large, nearly-equal numbers).

5. Mx, My, Mxy for the benchmark follow from the same expansion:
   w_xx = Sum C_kl phi_k''(x) phi_l(y), w_yy = Sum C_kl phi_k(x) phi_l''(y),
   w_xy = Sum C_kl phi_k'(x) phi_l'(y), then the usual
   Mx=-(w_xx+nu w_yy), My=-(w_yy+nu w_xx), Mxy=-(1-nu) w_xy (identical sign
   convention to the SS script and to compute_full_fields() below).

-----------------------------------------------------------------------------
WHAT ELSE CHANGES BECAUSE OF THE BC (both necessary consequences of FFFF,
not independent choices)
-----------------------------------------------------------------------------
- HARD-BC ENVELOPE: SS-SS-SS-SS used envelope = xi(1-xi)*eta(1-eta), which
  vanishes at each edge but has a nonzero derivative there (only w=0 is
  hard-enforced; Mx=0/My=0 at the edges was a SOFT loss term). FFFF needs
  BOTH w=0 and dw/dn=0 hard-enforced, so the envelope becomes
  envelope = xi^2*(1-xi)^2 * eta^2*(1-eta)^2 -- both the envelope and its
  first derivative vanish at xi,eta in {0,1} (standard clamped hard-BC
  trick), so the network's output satisfies FFFF exactly by construction,
  for any raw network output, everywhere in the domain.
- MOMENT LOSS REMOVED: SS-SS-SS-SS had a soft moment_loss enforcing Mx=0 /
  My=0 on the edges (the natural BC that Navier's series doesn't hard-
  enforce). FFFF has no such condition -- Mx, My are generally NONZERO at
  a clamped edge -- so there is nothing analogous to add back; both
  essential BCs (w=0, dw/dn=0) are already hard-enforced by the envelope
  above, so total_loss = pde_loss alone.
- The bc_loss diagnostic is kept (still ~0 by construction) and a new
  slope_bc_loss diagnostic is added (also ~0 by construction) as a sanity
  check that dw/dn is indeed ~0 on the edges, using the same central-
  difference stencil values already computed for the interior residual.

Everything else -- load, harmonic embedding, FD stencil, network backbone,
training loop, cost/diagnostic logging, plotting -- is copied unchanged
from the SS-SS-SS-SS FDM script.

Outputs (all written to ./results/<MODEL_NAME>/):
  - best_weights.pt, training_convergence.csv, cost_metrics.json,
    error_metrics.json, loss_balance.json, field_data.csv, figures/*.png

Run:
    python dh_fdm_pinn_triangular_hardBC_FFFF.py
"""


import os
import json
import time
import random

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from matplotlib import cm

# ============================================================
# CONFIG
# ============================================================
MODEL_NAME = "DH_FDM_PINN_Modal_HardBC_Triangular_FFFF"  # >>> VARIANT POINT: change per architecture
LOADING_CASE = "triangular"
BC_TYPE = "FFFF"            # all four edges clamped (w=0, dw/dn=0)
RESIDUAL_BACKBONE = "fdm"   # vs "autograd" elsewhere in this script family

# --- DH-PINN specific config -------------------------------------------
# Unchanged from the SS-SS-SS-SS script: these are input-feature choices
# driven by the LOAD's symmetry, not the BC, so they carry over as-is.
HARMONIC_INDICES_X = [1, 3, 5]   # odd only -- symmetric tent load in x
HARMONIC_INDICES_Y = [1, 3, 5]   # odd only -- still uniform in y
INCLUDE_RAW_COORDS = True
USE_PRODUCT_TERMS = True
GRAD_CLIP_NORM = 1.0
EMBEDDING_TYPE = "deterministic_harmonic_modal_odd" if USE_PRODUCT_TERMS else "deterministic_harmonic_odd"

# --- Ritz/Galerkin clamped-beam-function benchmark config ---------------
# Number of clamped-clamped beam eigenmodes per axis for the Rayleigh-Ritz
# analytical benchmark (see module docstring, part 3-4). K_MODES=10 is
# converged to ~5-6 significant figures against a known textbook clamped-
# plate coefficient (validated in chat) and stays well inside float64
# precision (see docstring point 4).
K_MODES = 10

OUT_DIR = os.path.join("results", MODEL_NAME)
FIG_DIR = os.path.join(OUT_DIR, "figures")
os.makedirs(FIG_DIR, exist_ok=True)

SEED = 42
MAX_EPOCHS = 15000
PATIENCE = 3000
PRINT_EVERY = 100
LR_INIT = 1e-3
LR_DECAY_STEPS = 2000
LR_DECAY_RATE = 0.95

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = torch.float64

# ============================================================
# SEEDING
# ============================================================
np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

# ============================================================
# PHYSICAL / NON-DIMENSIONAL PARAMETERS
# ============================================================
real_a, real_b, real_h = 1.0, 1.0, 0.01
real_E, real_nu, real_q0 = 200e9, 0.3, 1000.0

D_dim = real_E * (real_h ** 3) / (12 * (1 - real_nu ** 2))
L = real_a
W = real_q0 * (L ** 4) / D_dim
D_nd = 1.0
a_nd, b_nd, q0_nd, nu_nd = 1.0, 1.0, 1.0, real_nu

# ============================================================
# DOMAIN
# ============================================================
nx, ny = 51, 51
x_nd = np.linspace(0, a_nd, nx)
y_nd = np.linspace(0, b_nd, ny)
X_nd, Y_nd = np.meshgrid(x_nd, y_nd)   # 'xy' indexing -> shape (ny, nx)
grid_points_nd = np.vstack([X_nd.ravel(), Y_nd.ravel()]).T

X_grid = torch.tensor(grid_points_nd, dtype=DTYPE, device=DEVICE)

# --- FDM-PINN backbone: finite-difference step size for the PDE-residual
# stencil (unchanged from the SS-SS-SS-SS script -- see its docstring).
FD_H_X = a_nd / (nx - 1)
FD_H_Y = b_nd / (ny - 1)

center_x, center_y = a_nd / 2, b_nd / 2
center_idx = int(np.argmin((grid_points_nd[:, 0] - center_x) ** 2 +
                            (grid_points_nd[:, 1] - center_y) ** 2))

# --- Full boundary -- kept for the diagnostic bc_loss / slope_bc_loss ----
boundary_indices = np.unique(np.concatenate([
    np.where(grid_points_nd[:, 1] == 0)[0],
    np.where(grid_points_nd[:, 1] == b_nd)[0],
    np.where(grid_points_nd[:, 0] == 0)[0],
    np.where(grid_points_nd[:, 0] == a_nd)[0],
]))
interior_indices = np.setdiff1d(np.arange(nx * ny), boundary_indices)

bc_idx = torch.tensor(boundary_indices, dtype=torch.long, device=DEVICE)
int_idx = torch.tensor(interior_indices, dtype=torch.long, device=DEVICE)

# --- edges split by normal direction, for the slope diagnostic only -----
edge_x_indices = np.unique(np.concatenate([
    np.where(grid_points_nd[:, 0] == 0)[0],
    np.where(grid_points_nd[:, 0] == a_nd)[0],
]))   # dw/dx should be ~0 here
edge_y_indices = np.unique(np.concatenate([
    np.where(grid_points_nd[:, 1] == 0)[0],
    np.where(grid_points_nd[:, 1] == b_nd)[0],
]))   # dw/dy should be ~0 here
edge_x_idx = torch.tensor(edge_x_indices, dtype=torch.long, device=DEVICE)
edge_y_idx = torch.tensor(edge_y_indices, dtype=torch.long, device=DEVICE)

# ============================================================
# CLAMPED-CLAMPED BEAM EIGENFUNCTIONS (Rayleigh-Ritz basis for FFFF)
# See module docstring parts 1-2 for the derivation.
# ============================================================
def _solve_clamped_clamped_betas(n_modes):
    """Positive roots of cos(b)*cosh(b) = 1, via Newton's method seeded at
    (2n+1)*pi/2 (an excellent initial guess for every n -- verified against
    the standard tabulated roots 4.7300, 7.8532, 10.9956, 14.1372, ...)."""
    betas = []
    for n in range(1, n_modes + 1):
        beta = (2 * n + 1) * np.pi / 2
        for _ in range(200):
            c, s = np.cos(beta), np.sin(beta)
            ch, sh = np.cosh(beta), np.sinh(beta)
            f = c * ch - 1.0
            fp = -s * ch + c * sh
            step = f / fp
            beta -= step
            if abs(step) < 1e-15:
                break
        betas.append(beta)
    return np.array(betas)


_betas = _solve_clamped_clamped_betas(K_MODES)
_sigmas = (np.cosh(_betas) - np.cos(_betas)) / (np.sinh(_betas) - np.sin(_betas))


def _beam_phi(xi, b, s):
    return np.cosh(b * xi) - np.cos(b * xi) - s * (np.sinh(b * xi) - np.sin(b * xi))


def _beam_phi1(xi, b, s):
    return b * (np.sinh(b * xi) + np.sin(b * xi) - s * (np.cosh(b * xi) - np.cos(b * xi)))


def _beam_phi2(xi, b, s):
    return b ** 2 * (np.cosh(b * xi) + np.cos(b * xi) - s * (np.sinh(b * xi) + np.sin(b * xi)))


# --- Quadrature (fine trapezoidal grid -- smooth analytic integrands) ---
_xi_q = np.linspace(0.0, 1.0, 8001)
_phi_vals = [_beam_phi(_xi_q, _betas[i], _sigmas[i]) for i in range(K_MODES)]
_phi2_vals = [_beam_phi2(_xi_q, _betas[i], _sigmas[i]) for i in range(K_MODES)]

_N_modes = np.array([np.trapezoid(_phi_vals[i] ** 2, _xi_q) for i in range(K_MODES)])

_B_mat = np.zeros((K_MODES, K_MODES))
for _i in range(K_MODES):
    for _k in range(K_MODES):
        _B_mat[_i, _k] = np.trapezoid(_phi2_vals[_i] * _phi_vals[_k], _xi_q)

# --- Load modal projection: Q_kl = q0 * I_k * J_l (tent load separates) --
_tent_shape = np.where(_xi_q <= 0.5, 2.0 * _xi_q, 2.0 * (1.0 - _xi_q))  # peak 1 at xi=0.5
_I_modes = np.array([np.trapezoid(_tent_shape * _phi_vals[k], _xi_q) for k in range(K_MODES)])
_J_modes = np.array([np.trapezoid(_phi_vals[l], _xi_q) for l in range(K_MODES)])
_Q_kl = q0_nd * np.outer(_I_modes, _J_modes)

# --- Assemble and solve the dense K^2 x K^2 Galerkin system -------------
# C_kl*N_k*N_l*(b_k^4+b_l^4) + 2*Sum_ij B_ik*B_jl*C_ij = Q_kl  (docstring pt.3)
_D_kl = np.zeros((K_MODES, K_MODES))
for _k in range(K_MODES):
    for _l in range(K_MODES):
        _D_kl[_k, _l] = _N_modes[_k] * _N_modes[_l] * (_betas[_k] ** 4 + _betas[_l] ** 4)

_A_sys = (2 * np.einsum('ik,jl->klij', _B_mat, _B_mat).reshape(K_MODES * K_MODES, K_MODES * K_MODES)
          + np.diag(_D_kl.flatten()))
_C_ritz = np.linalg.solve(_A_sys, _Q_kl.flatten()).reshape(K_MODES, K_MODES)


def _clamped_series_field(x, y, kind="w"):
    """Evaluate the Ritz/Galerkin clamped-plate benchmark (w, Mx, My, or
    Mxy) on a field grid x, y, using the K_MODES x K_MODES beam-function
    expansion solved above."""
    result = np.zeros_like(x, dtype=np.float64)
    for k in range(K_MODES):
        bk, sk = _betas[k], _sigmas[k]
        fx = _beam_phi(x, bk, sk)
        fx1 = _beam_phi1(x, bk, sk)
        fx2 = _beam_phi2(x, bk, sk)
        for l in range(K_MODES):
            c_kl = _C_ritz[k, l]
            if c_kl == 0.0:
                continue
            bl, sl = _betas[l], _sigmas[l]
            fy = _beam_phi(y, bl, sl)
            if kind == "w":
                result += c_kl * fx * fy
            elif kind in ("Mx", "My"):
                fy2 = _beam_phi2(y, bl, sl)
                w_xx = c_kl * fx2 * fy
                w_yy = c_kl * fx * fy2
                if kind == "Mx":
                    result += -(w_xx + nu_nd * w_yy)
                else:
                    result += -(w_yy + nu_nd * w_xx)
            elif kind == "Mxy":
                fy1 = _beam_phi1(y, bl, sl)
                w_xy = c_kl * fx1 * fy1
                result += -(1 - nu_nd) * w_xy
            else:
                raise ValueError(kind)
    return result


def analytical_w_nd(x, y):
    return _clamped_series_field(x, y, "w")


def analytical_Mx_nd(x, y):
    return _clamped_series_field(x, y, "Mx")


def analytical_My_nd(x, y):
    return _clamped_series_field(x, y, "My")


def analytical_Mxy_nd(x, y):
    return _clamped_series_field(x, y, "Mxy")


def triangular_load_nd(x, y):
    """Symmetric tent load: 0 at x=0 and x=a, peak q0 at x=a/2, uniform in
    y. Unchanged from the SS-SS-SS-SS script -- x, y are torch tensors (y
    unused but kept for a consistent call signature)."""
    ramp_up = 2.0 * x / a_nd
    ramp_down = 2.0 * (a_nd - x) / a_nd
    q_x = torch.where(x <= a_nd / 2, ramp_up, ramp_down)
    return q0_nd * q_x


# ============================================================
# DETERMINISTIC HARMONIC EMBEDDING (unchanged from the SS-SS-SS-SS script)
# ============================================================
class HarmonicEmbedding(nn.Module):
    """
    gamma(xi, eta) = [xi, eta]  (optional, see INCLUDE_RAW_COORDS)
                      ++ [sin(k*pi*xi), cos(k*pi*xi)]_{k in harmonic_indices_x}
                      ++ [sin(k*pi*eta), cos(k*pi*eta)]_{k in harmonic_indices_y}
                      ++ [sin(m*pi*xi)*sin(n*pi*eta)]_{m in idx_x, n in idx_y}
    See the SS-SS-SS-SS script's docstring/class comment for the full
    rationale -- these are load-driven input features, independent of BC.
    """

    def __init__(self, harmonic_indices_x, harmonic_indices_y,
                 include_raw_coords: bool = True, use_product_terms: bool = True):
        super().__init__()
        self.harmonic_indices_x = list(harmonic_indices_x)
        self.harmonic_indices_y = list(harmonic_indices_y)
        n_x, n_y = len(self.harmonic_indices_x), len(self.harmonic_indices_y)
        self.include_raw_coords = include_raw_coords
        self.use_product_terms = use_product_terms
        kx = torch.tensor(self.harmonic_indices_x, dtype=DTYPE, device=DEVICE).view(1, -1)
        ky = torch.tensor(self.harmonic_indices_y, dtype=DTYPE, device=DEVICE).view(1, -1)
        self.register_buffer("kx", kx)
        self.register_buffer("ky", ky)

        out_dim = (2 if include_raw_coords else 0) + 2 * n_x + 2 * n_y
        if use_product_terms:
            out_dim += n_x * n_y
        self.out_dim = out_dim

    def forward(self, xi, eta):
        kx = self.kx * np.pi * xi
        ky = self.ky * np.pi * eta
        sin_x, cos_x = torch.sin(kx), torch.cos(kx)
        sin_y, cos_y = torch.sin(ky), torch.cos(ky)

        feats = [sin_x, cos_x, sin_y, cos_y]
        if self.include_raw_coords:
            feats = [xi, eta] + feats
        if self.use_product_terms:
            modal = sin_x.unsqueeze(2) * sin_y.unsqueeze(1)
            feats.append(modal.reshape(modal.shape[0], -1))
        return torch.cat(feats, dim=1)


# ============================================================
# MODEL (backbone identical to the SS-SS-SS-SS script; ONLY the hard-BC
# envelope changes, to hard-enforce w=0 AND dw/dn=0 for FFFF -- see
# module docstring)
# ============================================================
class PlatePINN(nn.Module):
    def __init__(self, hidden=64, n_layers=3,
                 harmonic_indices_x=HARMONIC_INDICES_X, harmonic_indices_y=HARMONIC_INDICES_Y,
                 include_raw_coords=INCLUDE_RAW_COORDS, use_product_terms=USE_PRODUCT_TERMS):
        super().__init__()
        self.embed = HarmonicEmbedding(harmonic_indices_x, harmonic_indices_y,
                                        include_raw_coords, use_product_terms)

        layers = []
        in_dim = self.embed.out_dim
        for _ in range(n_layers):
            layers.append(nn.Linear(in_dim, hidden, dtype=DTYPE))
            layers.append(nn.SiLU())
            in_dim = hidden
        self.hidden = nn.Sequential(*layers)
        self.out = nn.Linear(in_dim, 1, dtype=DTYPE)
        self.scale = nn.Parameter(torch.tensor(1.0, dtype=DTYPE))

    def forward(self, x, y):
        xi = x / a_nd
        eta = y / b_nd

        h = self.embed(xi, eta)
        h = self.hidden(h)
        raw = self.out(h)

        # Hard-constrained CLAMPED BC (w=0 AND dw/dn=0 on all edges):
        # envelope = xi^2(1-xi)^2 * eta^2(1-eta)^2. Both the envelope and
        # its first derivative vanish at xi,eta in {0,1} -- unlike the
        # SS-SS-SS-SS script's xi(1-xi)*eta(1-eta), which only killed the
        # envelope itself (order-1 zero), this is an order-2 zero at every
        # edge, so both essential BCs hold exactly regardless of `raw`.
        envelope = (xi ** 2) * (1 - xi) ** 2 * (eta ** 2) * (1 - eta) ** 2
        return self.scale * envelope * raw


model = PlatePINN().to(DEVICE)

# ============================================================
# OPTIMIZER + LR SCHEDULE
# ============================================================
optimizer = torch.optim.Adam(model.parameters(), lr=LR_INIT)
scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer, lr_lambda=lambda step: LR_DECAY_RATE ** (step / LR_DECAY_STEPS)
)


# ============================================================
# LOSS (4th-order derivatives via FINITE DIFFERENCES -- unchanged FDM
# backbone. No moment_loss for FFFF: both essential BCs are already
# hard-enforced by the envelope, so total_loss = pde_loss alone. bc_loss
# and slope_bc_loss are kept as diagnostics only (both ~0 by construction).
# ============================================================
def compute_loss():
    x = X_grid[:, 0:1]
    y = X_grid[:, 1:2]
    hx, hy = FD_H_X, FD_H_Y

    w_00 = model(x, y)
    w_p1_0 = model(x + hx, y)
    w_m1_0 = model(x - hx, y)
    w_p2_0 = model(x + 2 * hx, y)
    w_m2_0 = model(x - 2 * hx, y)
    w_0_p1 = model(x, y + hy)
    w_0_m1 = model(x, y - hy)
    w_0_p2 = model(x, y + 2 * hy)
    w_0_m2 = model(x, y - 2 * hy)
    w_p1_p1 = model(x + hx, y + hy)
    w_p1_m1 = model(x + hx, y - hy)
    w_m1_p1 = model(x - hx, y + hy)
    w_m1_m1 = model(x - hx, y - hy)

    w_xx = (w_p1_0 - 2 * w_00 + w_m1_0) / hx ** 2
    w_yy = (w_0_p1 - 2 * w_00 + w_0_m1) / hy ** 2

    w_xxxx = (w_p2_0 - 4 * w_p1_0 + 6 * w_00 - 4 * w_m1_0 + w_m2_0) / hx ** 4
    w_yyyy = (w_0_p2 - 4 * w_0_p1 + 6 * w_00 - 4 * w_0_m1 + w_0_m2) / hy ** 4
    w_xxyy = (w_p1_p1 - 2 * w_0_p1 + w_m1_p1
              - 2 * w_p1_0 + 4 * w_00 - 2 * w_m1_0
              + w_p1_m1 - 2 * w_0_m1 + w_m1_m1) / (hx ** 2 * hy ** 2)

    q = triangular_load_nd(x, y)

    pde = (w_xxxx + 2 * w_xxyy + w_yyyy - q)
    pde_loss = torch.mean(pde[int_idx] ** 2)

    # DIAGNOSTIC ONLY -- should be ~0 exactly, by construction of the
    # envelope (both terms).
    bc_loss = torch.mean(w_00[bc_idx] ** 2)
    w_x_c = (w_p1_0 - w_m1_0) / (2 * hx)
    w_y_c = (w_0_p1 - w_0_m1) / (2 * hy)
    slope_bc_loss = (torch.mean(w_x_c[edge_x_idx] ** 2)
                      + torch.mean(w_y_c[edge_y_idx] ** 2))

    total = pde_loss
    return total, pde_loss, bc_loss, slope_bc_loss, w_00


# ============================================================
# TRAINING LOOP with computational-cost logging + early stopping
# ============================================================
epoch_history, loss_history, pde_history, bc_history, slope_bc_history = [], [], [], [], []
w_center_history, epoch_time_history = [], []

best_loss = float("inf")
wait = 0
weights_path = os.path.join(OUT_DIR, "best_weights.pt")

if DEVICE.type == "cuda":
    torch.cuda.reset_peak_memory_stats()

train_start = time.perf_counter()
converged_epoch = MAX_EPOCHS

for epoch in range(MAX_EPOCHS):
    epoch_t0 = time.perf_counter()

    optimizer.zero_grad()
    total, pde, bc, slope_bc, w_pred = compute_loss()
    total.backward()
    if GRAD_CLIP_NORM is not None:
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
    optimizer.step()
    scheduler.step()

    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    epoch_dt = time.perf_counter() - epoch_t0

    loss_val = total.item()
    w_center_val = w_pred[center_idx, 0].item()

    epoch_history.append(epoch)
    loss_history.append(loss_val)
    pde_history.append(pde.item())
    bc_history.append(bc.item())
    slope_bc_history.append(slope_bc.item())
    w_center_history.append(w_center_val)
    epoch_time_history.append(epoch_dt)

    if epoch % PRINT_EVERY == 0:
        print(f"Epoch {epoch} Loss {loss_val:.6e} (pde {pde.item():.3e}) "
              f"hardBC_residual {bc.item():.3e} slope_residual {slope_bc.item():.3e} "
              f"Center {w_center_val:.6e} ({epoch_dt*1000:.2f} ms/epoch)")

    if loss_val < best_loss:
        best_loss = loss_val
        wait = 0
        torch.save(model.state_dict(), weights_path)
    else:
        wait += 1
        if wait >= PATIENCE:
            print(f"Early stopping triggered at epoch {epoch}")
            converged_epoch = epoch
            break

total_train_time = time.perf_counter() - train_start
peak_mem_mb = (torch.cuda.max_memory_allocated() / 1e6) if DEVICE.type == "cuda" else float("nan")

model.load_state_dict(torch.load(weights_path, map_location=DEVICE))
model.eval()

# ============================================================
# SAVE TRAINING CONVERGENCE CSV
# ============================================================
w_center_dim_history = np.array(w_center_history) * W
df_hist = pd.DataFrame({
    "epoch": epoch_history,
    "total_loss": loss_history,
    "pde_loss": pde_history,
    "hard_bc_residual_diagnostic": bc_history,
    "slope_bc_residual_diagnostic": slope_bc_history,
    "w_center_nd": w_center_history,
    "w_center_dim": w_center_dim_history,
    "epoch_time_s": epoch_time_history,
})
df_hist.to_csv(os.path.join(OUT_DIR, "training_convergence.csv"), index=False)
print("Saved training_convergence.csv")

# ============================================================
# LOSS DIAGNOSTIC (single-term loss for FFFF -- both essential BCs are
# hard-enforced, so there's no fraction-of-total split to report)
# ============================================================
final_pde, final_bc, final_slope_bc = pde_history[-1], bc_history[-1], slope_bc_history[-1]
loss_balance = {
    "model": MODEL_NAME,
    "loading_case": LOADING_CASE,
    "bc_type": BC_TYPE,
    "residual_backbone": RESIDUAL_BACKBONE,
    "fd_h_x": FD_H_X,
    "fd_h_y": FD_H_Y,
    "embedding_type": EMBEDDING_TYPE,
    "harmonic_indices_x": HARMONIC_INDICES_X,
    "harmonic_indices_y": HARMONIC_INDICES_Y,
    "k_modes_benchmark": K_MODES,
    "include_raw_coords": INCLUDE_RAW_COORDS,
    "use_product_terms": USE_PRODUCT_TERMS,
    "grad_clip_norm": GRAD_CLIP_NORM,
    "final_pde_loss": final_pde,
    "final_hard_bc_residual_diagnostic": final_bc,
    "final_slope_bc_residual_diagnostic": final_slope_bc,
    "note": "total_loss == pde_loss for FFFF; w=0 and dw/dn=0 are hard-enforced by the envelope.",
}
if final_bc > 1e-20 or final_slope_bc > 1e-20:
    print(f"WARNING: hard_bc_residual_diagnostic = {final_bc:.3e} / "
          f"slope_bc_residual_diagnostic = {final_slope_bc:.3e} not ~0 -- "
          f"double-check the envelope function / grid point exactness (float64 expected).")
with open(os.path.join(OUT_DIR, "loss_balance.json"), "w") as f:
    json.dump(loss_balance, f, indent=2)
print("Saved loss_balance.json:", loss_balance)

# ============================================================
# COMPUTATIONAL COST METRICS
# ============================================================
cost_metrics = {
    "model": MODEL_NAME,
    "loading_case": LOADING_CASE,
    "bc_type": BC_TYPE,
    "residual_backbone": RESIDUAL_BACKBONE,
    "fd_h_x": FD_H_X,
    "fd_h_y": FD_H_Y,
    "embedding_type": EMBEDDING_TYPE,
    "harmonic_indices_x": HARMONIC_INDICES_X,
    "harmonic_indices_y": HARMONIC_INDICES_Y,
    "include_raw_coords": INCLUDE_RAW_COORDS,
    "use_product_terms": USE_PRODUCT_TERMS,
    "grad_clip_norm": GRAD_CLIP_NORM,
    "input_feature_dim": model.embed.out_dim,
    "device": str(DEVICE),
    "epochs_run": len(epoch_history),
    "epochs_to_early_stop": converged_epoch,
    "epochs_to_convergence": converged_epoch,
    "mean_time_per_epoch_s": float(np.mean(epoch_time_history)),
    "median_time_per_epoch_s": float(np.median(epoch_time_history)),
    "total_training_time_s": total_train_time,
    "peak_gpu_memory_mb": peak_mem_mb,
    "n_trainable_params": sum(p.numel() for p in model.parameters() if p.requires_grad),
}
with open(os.path.join(OUT_DIR, "cost_metrics.json"), "w") as f:
    json.dump(cost_metrics, f, indent=2)
print("Saved cost_metrics.json:", cost_metrics)

# ============================================================
# POST-PROCESSING: full-field predictions (w, Mx, My) via autograd
# (unchanged -- still differentiates the trained network exactly, isolated
# from the training-time FD stencil, same rationale as the SS script)
# ============================================================
def compute_full_fields():
    x = X_grid[:, 0:1].clone().requires_grad_(True)
    y = X_grid[:, 1:2].clone().requires_grad_(True)
    w = model(x, y)
    ones = torch.ones_like(w)

    w_x = torch.autograd.grad(w, x, grad_outputs=ones, create_graph=True)[0]
    w_y = torch.autograd.grad(w, y, grad_outputs=ones, create_graph=True)[0]
    w_xx = torch.autograd.grad(w_x, x, grad_outputs=ones, create_graph=True)[0]
    w_yy = torch.autograd.grad(w_y, y, grad_outputs=ones, create_graph=True)[0]
    w_xy = torch.autograd.grad(w_x, y, grad_outputs=ones, create_graph=True)[0]

    Mx = -(w_xx + nu_nd * w_yy)
    My = -(w_yy + nu_nd * w_xx)
    Mxy = -(1 - nu_nd) * w_xy

    w_grid = w.detach().cpu().numpy().reshape(ny, nx)
    Mx_grid = Mx.detach().cpu().numpy().reshape(ny, nx)
    My_grid = My.detach().cpu().numpy().reshape(ny, nx)
    Mxy_grid = Mxy.detach().cpu().numpy().reshape(ny, nx)
    return w_grid, Mx_grid, My_grid, Mxy_grid


w_pred_nd, Mx_pred_nd, My_pred_nd, Mxy_pred_nd = compute_full_fields()

w_anal_nd = analytical_w_nd(X_nd, Y_nd)
Mx_anal_nd = analytical_Mx_nd(X_nd, Y_nd)
My_anal_nd = analytical_My_nd(X_nd, Y_nd)
Mxy_anal_nd = analytical_Mxy_nd(X_nd, Y_nd)

w_dim_pred, w_dim_anal = w_pred_nd * W, w_anal_nd * W
moment_scale = D_dim * W / L ** 2
Mx_dim_pred, Mx_dim_anal = Mx_pred_nd * moment_scale, Mx_anal_nd * moment_scale
My_dim_pred, My_dim_anal = My_pred_nd * moment_scale, My_anal_nd * moment_scale
Mxy_dim_pred, Mxy_dim_anal = Mxy_pred_nd * moment_scale, Mxy_anal_nd * moment_scale

w_signed_error = w_dim_pred - w_dim_anal
w_error = np.abs(w_signed_error)
Mx_error = np.abs(Mx_dim_pred - Mx_dim_anal)
My_error = np.abs(My_dim_pred - My_dim_anal)
Mxy_error = np.abs(Mxy_dim_pred - Mxy_dim_anal)

# ============================================================
# ERROR METRICS
# ============================================================
ix, iy = nx // 2, ny // 2
w_pred_center, w_anal_center = w_dim_pred[iy, ix], w_dim_anal[iy, ix]
center_abs_err = abs(w_pred_center - w_anal_center)
center_pct_err = center_abs_err / abs(w_anal_center) * 100

L2_error_w = np.sqrt(np.mean((w_dim_pred - w_dim_anal) ** 2))
rel_L2_error_w = np.linalg.norm(w_dim_pred - w_dim_anal) / np.linalg.norm(w_dim_anal)

error_metrics = {
    "model": MODEL_NAME,
    "loading_case": LOADING_CASE,
    "bc_type": BC_TYPE,
    "residual_backbone": RESIDUAL_BACKBONE,
    "embedding_type": EMBEDDING_TYPE,
    "harmonic_indices_x": HARMONIC_INDICES_X,
    "harmonic_indices_y": HARMONIC_INDICES_Y,
    "use_product_terms": USE_PRODUCT_TERMS,
    "k_modes_benchmark": K_MODES,
    "center_deflection_pred_m": float(w_pred_center),
    "center_deflection_anal_m": float(w_anal_center),
    "center_abs_error_m": float(center_abs_err),
    "center_pct_error": float(center_pct_err),
    "w_max_abs_error_m": float(np.max(w_error)),
    "w_rms_error_m": float(np.sqrt(np.mean(w_error ** 2))),
    "w_L2_error_m": float(L2_error_w),
    "w_relative_L2_error": float(rel_L2_error_w),
    "Mx_max_abs_error_Nm": float(np.max(Mx_error)),
    "Mx_rms_error_Nm": float(np.sqrt(np.mean(Mx_error ** 2))),
    "My_max_abs_error_Nm": float(np.max(My_error)),
    "My_rms_error_Nm": float(np.sqrt(np.mean(My_error ** 2))),
    "Mxy_max_abs_error_Nm": float(np.max(Mxy_error)),
    "Mxy_rms_error_Nm": float(np.sqrt(np.mean(Mxy_error ** 2))),
    "boundary_w_max_abs_value_m": float(np.max(np.abs(w_dim_pred[0, :]))
                                         + np.max(np.abs(w_dim_pred[-1, :]))
                                         + np.max(np.abs(w_dim_pred[:, 0]))
                                         + np.max(np.abs(w_dim_pred[:, -1]))),
}
with open(os.path.join(OUT_DIR, "error_metrics.json"), "w") as f:
    json.dump(error_metrics, f, indent=2)
print("Saved error_metrics.json:", error_metrics)

df_field = pd.DataFrame({
    "x_nd": X_nd.ravel(), "y_nd": Y_nd.ravel(),
    "w_dim_anal": w_dim_anal.ravel(), "w_dim_pred": w_dim_pred.ravel(),
    "Mx_dim_anal": Mx_dim_anal.ravel(), "Mx_dim_pred": Mx_dim_pred.ravel(),
    "My_dim_anal": My_dim_anal.ravel(), "My_dim_pred": My_dim_pred.ravel(),
    "Mxy_dim_anal": Mxy_dim_anal.ravel(), "Mxy_dim_pred": Mxy_dim_pred.ravel(),
})
df_field["w_abs_error"] = np.abs(df_field["w_dim_anal"] - df_field["w_dim_pred"])
df_field["w_rel_error"] = df_field["w_abs_error"] / (np.abs(df_field["w_dim_anal"]) + 1e-12)
df_field.to_csv(os.path.join(OUT_DIR, "field_data.csv"), index=False)
print("Saved field_data.csv")

# ============================================================
# PLOTTING
# ============================================================
plt.rcParams.update({"figure.dpi": 150})


def save_contour(field, title, cbar_label, fname, cmap=cm.jet):
    plt.figure(figsize=(7, 6))
    cf = plt.contourf(X_nd, Y_nd, field, levels=50, cmap=cmap)
    plt.colorbar(cf, label=cbar_label)
    plt.title(title, fontstyle="italic", fontsize=13)
    plt.xlabel("x/a", fontstyle="italic")
    plt.ylabel("y/b", fontstyle="italic")
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, fname))
    plt.close()


plt.figure(figsize=(8, 5))
plt.plot(df_hist["epoch"], df_hist["w_center_dim"], linewidth=2, label=f"{MODEL_NAME} Prediction")
plt.axhline(y=w_anal_center, linestyle="--", linewidth=2, color="k", label="Ritz/Galerkin Benchmark")
plt.xlabel("Epoch"); plt.ylabel("Center Deflection w (m)")
plt.title("Convergence of Center Deflection")
plt.legend(); plt.grid(True, linestyle="--", alpha=0.6); plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "Convergence_Center_Deflection.png")); plt.close()

plt.figure(figsize=(8, 5))
plt.plot(df_hist["epoch"], np.abs(df_hist["w_center_dim"] - w_anal_center), linewidth=2)
plt.yscale("log")
plt.xlabel("Epoch"); plt.ylabel("Absolute Error at Center (m)")
plt.title("Convergence Error (Log Scale)")
plt.grid(True, which="both", linestyle="--", alpha=0.6); plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "Convergence_Error_Log_Scale.png")); plt.close()

plt.figure(figsize=(8, 6))
plt.plot(df_hist["epoch"], df_hist["total_loss"], label="Total Loss (= PDE Loss)")
plt.plot(df_hist["epoch"], df_hist["hard_bc_residual_diagnostic"],
         label="Hard-BC residual (diagnostic, ~0)", alpha=0.6, linestyle=":")
plt.plot(df_hist["epoch"], df_hist["slope_bc_residual_diagnostic"],
         label="Slope-BC residual (diagnostic, ~0)", alpha=0.6, linestyle=":")
plt.yscale("log")
plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.title("Training Loss Curve")
plt.legend(); plt.grid(True, which="both", ls="--", alpha=0.7); plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "Training_Loss_Curve.png")); plt.close()

save_contour(w_dim_anal, "Ritz/Galerkin Benchmark Plate Deflection", "Deflection (m)",
             "Analytical_Plate_Deflection.png")
save_contour(w_dim_pred, f"{MODEL_NAME} Predicted Plate Deflection", "Deflection (m)",
             "PINN_Predicted_Plate_Deflection.png")
save_contour(w_error, "Absolute Error in Plate Deflection", "|w_pred - w_anal| (m)",
             "Absolute_Error_in_Plate_Deflection.png", cmap=cm.inferno)
save_contour(w_signed_error, "Signed Error w_pred - w_anal", "w_pred - w_anal (m)",
             "Absolute_Error_w_pred_-_w_anal.png", cmap="coolwarm")

save_contour(Mx_dim_anal, "Ritz/Galerkin Benchmark Bending Moment $M_x$", "Mx (N·m/m)",
             "Bending_Moment_M_x_Analytical.png")
save_contour(Mx_dim_pred, "Predicted Bending Moment $M_x$", "Mx (N·m/m)",
             "Bending_Moment_M_x.png")
save_contour(Mx_error, "Absolute Error in $M_x$", "|Mx_pred - Mx_anal| (N·m/m)",
             "Absolute_Error_in_M_x.png", cmap="hot")

save_contour(My_dim_anal, "Ritz/Galerkin Benchmark Bending Moment $M_y$", "My (N·m/m)",
             "Bending_Moment_M_y_Analytical.png")
save_contour(My_dim_pred, "Predicted Bending Moment $M_y$", "My (N·m/m)",
             "Bending_Moment_M_y.png")
save_contour(My_error, "Absolute Error in $M_y$", "|My_pred - My_anal| (N·m/m)",
             "Absolute_Error_in_M_y.png", cmap="hot")

save_contour(Mxy_dim_anal, "Ritz/Galerkin Benchmark Twisting Moment $M_{xy}$", "Mxy (N·m/m)",
             "Twisting_Moment_M_xy_Analytical.png")
save_contour(Mxy_dim_pred, "Predicted Twisting Moment $M_{xy}$", "Mxy (N·m/m)",
             "Twisting_Moment_M_xy.png")
save_contour(Mxy_error, "Absolute Error in $M_{xy}$", "|Mxy_pred - Mxy_anal| (N·m/m)",
             "Absolute_Error_in_M_xy.png", cmap="hot")

plt.figure(figsize=(6, 6))
plt.scatter(w_dim_anal.ravel(), w_dim_pred.ravel(), s=8, alpha=0.5)
lims = [min(w_dim_anal.min(), w_dim_pred.min()), max(w_dim_anal.max(), w_dim_pred.max())]
plt.plot(lims, lims, "k--", linewidth=1, label="Perfect agreement")
plt.xlabel("Ritz/Galerkin benchmark displacement w (m)", fontstyle="italic")
plt.ylabel("Predicted displacement w (m)", fontstyle="italic")
plt.title("Benchmark vs Predicted Displacement", fontstyle="italic")
plt.legend(); plt.grid(True); plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "Analytical_vs_Predicted_Displacement.png")); plt.close()

err_fft = np.fft.fftshift(np.fft.fft2(w_error))
err_fft_mag = np.log1p(np.abs(err_fft))
plt.figure(figsize=(7, 6))
plt.imshow(err_fft_mag, cmap="magma", extent=[-nx // 2, nx // 2, -ny // 2, ny // 2])
plt.colorbar(label="log(1 + |FFT error|)")
plt.title(f"Error Spectrum (FFT) — {LOADING_CASE}, {BC_TYPE}")
plt.xlabel("kx"); plt.ylabel("ky")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "Error_Spectrum_FFT.png")); plt.close()

fig, axes = plt.subplots(4, 3, figsize=(16, 18))
rows = [
    ("Displacement w (m)", w_dim_anal, w_dim_pred, w_error),
    ("Moment Mx (N·m)", Mx_dim_anal, Mx_dim_pred, Mx_error),
    ("Moment My (N·m)", My_dim_anal, My_dim_pred, My_error),
    ("Moment Mxy (N·m)", Mxy_dim_anal, Mxy_dim_pred, Mxy_error),
]
for r, (label, anal, pred, err) in enumerate(rows):
    for c, (title, field, cmap) in enumerate([
        (f"Benchmark {label}", anal, cm.jet),
        (f"Predicted {label}", pred, cm.jet),
        (f"Absolute Error {label}", err, cm.inferno),
    ]):
        ax = axes[r, c]
        cf = ax.contourf(X_nd, Y_nd, field, levels=20, cmap=cmap)
        fig.colorbar(cf, ax=ax)
        ax.set_title(title, fontsize=10)
        ax.set_xlabel("x/a"); ax.set_ylabel("y/b")
plt.suptitle(f"{MODEL_NAME} — {LOADING_CASE} loading, {BC_TYPE} (summary grid)", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "_summary_grid_all_fields.png")); plt.close()

print(f"\nAll outputs written to: {os.path.abspath(OUT_DIR)}")
print(f"Total training time: {total_train_time:.2f}s | "
      f"Epochs to early-stop: {converged_epoch} | "
      f"Peak GPU mem: {peak_mem_mb:.1f} MB" if DEVICE.type == "cuda" else
      f"\nAll outputs written to: {os.path.abspath(OUT_DIR)} | "
      f"Total training time: {total_train_time:.2f}s | Epochs: {converged_epoch}")

Epoch 0 Loss 3.841276e+00 (pde 3.841e+00) hardBC_residual 0.000e+00 slope_residual 1.539e-11 Center -5.055540e-04 (502.61 ms/epoch)
Epoch 100 Loss 5.488375e-02 (pde 5.488e-02) hardBC_residual 0.000e+00 slope_residual 2.030e-10 Center 7.702234e-04 (56.43 ms/epoch)
Epoch 200 Loss 3.978463e-02 (pde 3.978e-02) hardBC_residual 0.000e+00 slope_residual 2.047e-10 Center 7.672404e-04 (55.79 ms/epoch)
Epoch 300 Loss 3.750407e-02 (pde 3.750e-02) hardBC_residual 0.000e+00 slope_residual 2.088e-10 Center 7.677283e-04 (54.64 ms/epoch)
Epoch 400 Loss 3.550058e-02 (pde 3.550e-02) hardBC_residual 0.000e+00 slope_residual 2.099e-10 Center 7.658723e-04 (55.55 ms/epoch)
Epoch 500 Loss 3.394080e-02 (pde 3.394e-02) hardBC_residual 0.000e+00 slope_residual 2.111e-10 Center 7.680504e-04 (58.31 ms/epoch)
Epoch 600 Loss 3.255453e-02 (pde 3.255e-02) hardBC_residual 0.000e+00 slope_residual 2.121e-10 Center 7.695972e-04 (55.56 ms/epoch)
Epoch 700 Loss 3.149954e-02 (pde 3.150e-02) hardBC_residual 0.000e+00 slope_

C:\Users\vivek\AppData\Local\Temp\ipykernel_15228\733643985.py:620: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(weights_path, map_location

Saved training_convergence.csv
Saved loss_balance.json: {'model': 'DH_FDM_PINN_Modal_HardBC_Triangular_FFFF', 'loading_case': 'triangular', 'bc_type': 'FFFF', 'residual_backbone': 'fdm', 'fd_h_x': 0.02, 'fd_h_y': 0.02, 'embedding_type': 'deterministic_harmonic_modal_odd', 'harmonic_indices_x': [1, 3, 5], 'harmonic_indices_y': [1, 3, 5], 'k_modes_benchmark': 10, 'include_raw_coords': True, 'use_product_terms': True, 'grad_clip_norm': 1.0, 'final_pde_loss': 1.2298422536736848e-05, 'final_hard_bc_residual_diagnostic': 0.0, 'final_slope_bc_residual_diagnostic': 2.70929194553067e-10, 'note': 'total_loss == pde_loss for FFFF; w=0 and dw/dn=0 are hard-enforced by the envelope.'}
Saved cost_metrics.json: {'model': 'DH_FDM_PINN_Modal_HardBC_Triangular_FFFF', 'loading_case': 'triangular', 'bc_type': 'FFFF', 'residual_backbone': 'fdm', 'fd_h_x': 0.02, 'fd_h_y': 0.02, 'embedding_type': 'deterministic_harmonic_modal_odd', 'harmonic_indices_x': [1, 3, 5], 'harmonic_indices_y': [1, 3, 5], 'include_ra